## Installing sentence-transformers

Install the library:

In [1]:
!uv add sentence-transformers

Resolved 168 packages in 971ms                                       
⠙ Preparing packages... (0/13)                                                  ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/13)------------------     0 B/116.45 KiB          
⠙ Preparing packages... (0/13)------------------ 14.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)------------------ 30.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)------------------ 46.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)-------------- 62.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)---------- 78.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)--------- 94.81 KiB/116.45 KiB        
⠙ Preparing packages... (0/13)--------- 110.81 KiB/116.45 KiB       
⠙ Preparing packages... (0/13)--------- 116.45 KiB/116.45 KiB       
⠙ Preparing packages... (0/13)--------- 116.45 KiB/116.45 KiB       
markdown-it-py       ----------------

## Choosing a model

Sentence-transformers supports many models. The right one depends on
your task, your language, and the resources you have. Larger models are
usually slower, so for our FAQ dataset of short English texts a small
model is enough. Try a few on your own data and keep the one that works
best.

We'll use `all-MiniLM-L6-v2`:

- 384-dimensional vectors (compact)
- Fast on CPU
- Good quality for general English text
- Uses cosine similarity (we'll explain this below)

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Trying it with simple examples

Let's see how embeddings work on a few examples.

We'll start with a query:

In [3]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"

In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

`v1` is a vector, an array of 384 numbers. Each number stands for some
concept the model learned. We can't read off what any one of them means.
But two vectors with similar values point to texts about similar things.

Encode our document:

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
dv

array([-4.81207073e-02, -1.29942060e-01, -4.86809807e-03,  1.37495305e-02,
       -8.51100218e-03,  6.97840899e-02, -7.55662099e-03,  4.09572423e-02,
       -1.02165677e-01, -3.45050804e-02,  2.85520982e-02, -5.81431054e-02,
       -1.85112990e-02, -3.17838676e-02,  5.29631376e-02,  3.96292843e-02,
       -9.84600093e-03, -6.45324364e-02,  7.05098212e-02, -6.39589429e-02,
       -3.55156921e-02, -1.75489821e-02, -3.58401500e-02,  5.50837838e-04,
        3.38445753e-02,  2.91763581e-02,  3.24279219e-02, -5.64804487e-03,
        8.42592567e-02,  5.90307005e-02,  2.08771937e-02, -5.44836977e-03,
        2.98404042e-02,  1.30820274e-02,  6.36078343e-02, -3.06394007e-02,
        6.28460646e-02, -1.69217698e-02, -4.65758666e-02, -5.13761975e-02,
       -3.81561071e-02, -7.01197386e-02, -9.55517739e-02,  5.01639768e-02,
        8.57904460e-03,  1.19207595e-02, -1.40597112e-02, -7.98978098e-03,
       -3.44239138e-02, -2.34269761e-02,  2.89466120e-02,  2.68904865e-02,
       -4.87409085e-02, -

In [8]:
v1.dot(dv)

np.float32(0.32332402)

## Loading the data
Use the `ingest.py` from module 1 or loading the FAQ data.

In [9]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py


--2026-06-27 19:01:48--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-27 19:01:48 (42.3 MB/s) - ‘ingest.py’ saved [888/888]



In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

## Generating embeddings

Each document is a Python dictionary with a question and an answer. We
embed both together. That way a query can match against the question
text and the answer text in our index.

Build one text per document:

In [12]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [16]:
texts.__len__()

1350

Now we generate the embeddings. We have about 1200 texts here. We won't
hand the model all of them at once. That takes a long time, and we can't
see what's happening inside. Instead we split them into batches.

First we import `tqdm` so we can watch the progress:

In [17]:
from tqdm.auto import tqdm


Next we chunk the dataset into batches of 50 and encode each batch:


In [18]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [20]:
vectors[10].shape

(384,)

In [21]:
v1.dot(vectors[10])

np.float32(0.33153287)

We end up with ≈1208 vectors. On a GPU this is fast. Most of us run on
Codespaces without a GPU, so it takes a bit, but it's a one-off.

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

## Scoring documents
This is matrix-vector multiplication. Each element `i` of `scores` is
the cosine similarity between document `i` (row `i` of `X`) and
`v_query`.

We could compute the same thing with a for loop:

In [23]:
import numpy as np
X = np.array(vectors)

In [ ]:
scores = [dv.dot(X[i]) for i in range(len(X))] #(?) from notes

In [28]:
scores = X.dot(v1)

## Best match

The highest score is the most similar document:

In [29]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

This returns document 553 with score 0.76.

The index and score may differ for you. Our FAQ is a living document, so
we add and remove entries over time.

Let's see which document it is:

In [30]:
documents[553]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'OpenAI: Error when running OpenAI responses.create command',
 'answer': 'You may receive the following error when running the OpenAI `responses.create` command due to insufficient credits in your OpenAI account:\n\n```\nOpenAI API Error: Insufficient credits\n```',
 'doc_id': 'f5df151c59'}

## Top 5 results
Usually we want more than the single best match, so let's pull the top
5.

`np.argsort` sorts from lowest to highest, so the last 5 are the top
ones:

In [31]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

There's a shorter trick I usually reach for. We negate the scores
first, so the largest becomes the smallest. Then `argsort` puts the best
matches at the front.

Here it is in one line:

In [33]:
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

There's a shorter trick I usually reach for. We negate the scores
first, so the largest becomes the smallest. Then `argsort` puts the best
matches at the front.

Here it is in one line:

In [34]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192134
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions',

# Vector Search with minsearch

## Creating the index

We already have our documents and vectors from the previous section.

Index them:

In [36]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [39]:
vindex.search(v1, num_results=5, filter_dict={'course':'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

# RAG with Vector Search
The search step used keyword search. Now we swap in vector search.
Because RAG is modular, search is the only step we touch. Build prompt
and the LLM call stay exactly as they were.

## Using RAGBase
First, create the OpenAI client:

In [41]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Download `rag_helper.py`

In [40]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-27 20:37:52--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8003::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-27 20:37:52 (12.3 MB/s) - ‘rag_helper.py’ saved [2134/2134]



Next, download and index the data:

In [42]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [44]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes — you can still join. If you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [46]:
assistant.search(query)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.',
  'doc_id': '9f689c185f'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Do I need a new GitHub repo for Module 2, or just a new codespace?',
  'answer': "Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you creat


This still uses keyword search. Text search isn't bad here, so the
answer may already look right. Next we replace search with vector
search.

We already have:

- All the indexed documents `documents`
- The embeddings matrix `X` with all these documents
- The vector search engine `vindex`

We can't pass `vindex` to RAG as-is. Text search takes the query string
directly, but vector search needs the query as a vector first. So we
subclass `RAGBase` and override `search` to encode the query before
searching.

The subclass overrides `search`:

In [48]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

The `__init__` method adds one extra argument, `embedder`, for the
sentence transformer. Inside `search` we use it to turn the query into a
vector. Then we query `vindex` with that vector instead of the raw text.
Everything else is inherited from `RAGBase`.

In [49]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [50]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

The answers should be close to what we got with keyword search, but
vector search handles rephrased questions better. The swap was trivial
because RAG has three clear steps. The same trick lets us change the LLM
provider later by overriding just the `llm` step.

## sqlitesearch


sqlitesearch is the persistent sibling of minsearch, and it solves both
problems at once.

We already used it in module 1 for persistent text search. It also does
vector search through its `VectorSearchIndex` class. It stores vectors
in SQLite, a real on-disk database, and uses ANN strategies for
retrieval. Because the data lives on disk, one process can write the
vectors and another can read them back.


sqlitesearch supports three ANN modes:

- `lsh` (default): up to 100K vectors, random hyperplane projections
- `ivf`: 10K-500K vectors, K-means clustering
- `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)

For our small dataset, `lsh` is fine. All modes use two-phase search:
approximate candidate retrieval, then exact cosine similarity
reranking.


In [51]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

## Indexing the data

Fit the index with our vectors and documents:


In [52]:
vs_index.fit(vectors, documents)

The index is saved to `faq_vectors2.db`. Unlike minsearch, this file
persists on disk. You can search immediately after indexing, or reopen
the index later without re-indexing.

## Searching
Search works the same way as with minsearch. We always encode the query
into a vector first. This is one thing that makes vector search heavier
than text search. With text search we'd throw the raw query straight at
the engine.

Encode, then search:

In [53]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

Look at the results:



In [55]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

## Closing the connection

When you're done with the index:

In [54]:
vs_index.close()

## Filtering by course

Filtering works the same way:


In [56]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere